# System C — Dynamic factor cross-sport model (beginner version)

**Goal:** represent each fighter as a *latent state* that can be updated by multiple sports.

Why you need this:
- A kickboxing bout should primarily update **striking**, not grappling.
- A grappling match should primarily update **grappling**.
- MMA depends on both striking and grappling (and optionally durability/cardio).
- After a non-MMA bout, the fighter entering the next MMA bout is not “the same” as before.

This notebook explains a simple and scalable way to model that idea.

---

## 1) The latent state (what we store per fighter)

We store a vector per fighter:

$$
\theta_i =
\begin{bmatrix}
\theta^{mma}_i \\
\theta^{str}_i \\
\theta^{grp}_i \\
\theta^{dur}_i
\end{bmatrix}
$$

- **MMA**: overall MMA-specific skill beyond just strike/grapple
- **str**: striking skill
- **grp**: grappling skill
- **dur**: durability/cardio proxy (optional but useful)

---

## 2) Sports “look at” different parts of the vector

Each sport $s$ has a weight vector $w_s$:

- MMA: $w_{mma} = [1, a_s, a_g, a_d]$
- Kickboxing: $w_{kb} = [0, 1, 0, a_d^{kb}]$
- Grappling: $w_{gr} = [0, 0, 1, a_d^{gr}]$

Then a bout uses a paired-comparison model:

$$
z = w_s^\top(\theta_i - \theta_j)
$$
$$
p(i\text{ wins}) = \sigma(z)
$$

where $\sigma(\cdot)$ is the logistic function.

---

## 3) Time dynamics (ring rust / drift)

Between bouts, uncertainty should grow.
In a full Bayesian filter you track covariance; in a simpler online model you can:
- increase learning rate after long layoffs,
- or add a “mean reversion” / drift term.

We'll show a simple demonstration.

---

## 4) Tier A/B/C integration (where PS helps)

For **MMA bouts**:
- Tier A/B gives you additional signals about *how* the fight went (PS margin).
- System C can ingest that as an extra feature or as a second observation channel (finish intensity).

For **non-MMA bouts**:
- you usually only have Tier C (outcome/method).
- updates should be conservative, especially for low-volume sports (strong shrinkage).


In [ ]:
import sys
from dataclasses import replace
from datetime import date
from importlib import import_module
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

system_c_module = import_module('elo_calculator.application.ranking.system_c_dynamic_factor_bt')
types_module = import_module('elo_calculator.application.ranking.types')
DynamicFactorBradleyTerrySystem = system_c_module.DynamicFactorBradleyTerrySystem
new_dynamic_factor_state = system_c_module.new_dynamic_factor_state
BoutEvidence = types_module.BoutEvidence
BoutOutcome = types_module.BoutOutcome
EvidenceTier = types_module.EvidenceTier
FinishMethod = types_module.FinishMethod

## 5) Toy example: kickboxing improves striking, which changes the next MMA prior

We will simulate:

1) Fighter A vs Fighter B in kickboxing (A wins)
2) Then Fighter A vs Fighter C in MMA

We'll compare MMA win probability *before* and *after* the kickboxing update.

This shows the “cross-sport gap effect”:
- intervening non-MMA bouts update latent attributes,
- which changes the prior for the next MMA bout.


In [ ]:
system = DynamicFactorBradleyTerrySystem()

state_a = new_dynamic_factor_state('a')
state_b = new_dynamic_factor_state('b')
state_c = replace(new_dynamic_factor_state('c'), mean=np.array([0.0, 0.0, 0.4, 0.0], dtype=np.float64))

mma_probe_pre = BoutEvidence(
    bout_id='probe_pre',
    event_date=date(2024, 1, 1),
    sport_key='mma',
    outcome_for_a=BoutOutcome.WIN,
    tier=EvidenceTier.C,
    method=FinishMethod.DEC,
)

kb_bout = BoutEvidence(
    bout_id='kb_1',
    event_date=date(2024, 2, 1),
    sport_key='kickboxing',
    outcome_for_a=BoutOutcome.WIN,
    tier=EvidenceTier.C,
    method=FinishMethod.DEC,
)

mma_probe_post = BoutEvidence(
    bout_id='probe_post',
    event_date=date(2024, 3, 1),
    sport_key='mma',
    outcome_for_a=BoutOutcome.WIN,
    tier=EvidenceTier.C,
    method=FinishMethod.DEC,
)

probability_before = system.expected_win_probability(state_a, state_c, mma_probe_pre)
state_a_after_kb, state_b_after_kb, delta_kb_a, _ = system.update_bout(state_a, state_b, kb_bout)
probability_after = system.expected_win_probability(state_a_after_kb, state_c, mma_probe_post)

rows = [
    {'metric': 'mma_prob_a_vs_c_before_kb', 'value': round(probability_before, 5)},
    {'metric': 'kickboxing_update_delta_rating_a', 'value': round(delta_kb_a.delta_rating, 5)},
    {'metric': 'mma_prob_a_vs_c_after_kb', 'value': round(probability_after, 5)},
    {'metric': 'probability_shift_after_kb', 'value': round(probability_after - probability_before, 5)},
]

pd.DataFrame(rows)

## 6) How to make this production-grade

The demo above is *not* a full Bayesian filter; it's a readable approximation.

A production implementation typically adds:

### 6.1 Uncertainty / covariance
Store a diagonal (or low-rank) covariance matrix per fighter:
- uncertainty grows with inactivity,
- updates shrink when uncertainty is low.

This makes behavior closer to TrueSkill / Kalman filtering.

### 6.2 Sport transfer shrinkage
Because many sports have low volume, updates should be conservative.
Typical approach:
- lower learning rates (or higher priors) for low-volume sports,
- hierarchical priors on the sport weight vectors $w_s$.

### 6.3 MMA-only leaderboard
Even though the model uses all sports, you serve an **MMA ranking** by sorting on:
- $\theta^{mma}$ (or a linear combination like $w_{mma}^\top\theta$)
with an uncertainty penalty if desired.

### 6.4 Tier A PS as extra evidence
On UFCStats fights, you can add:
- a second channel modeling finish intensity,
- or a continuous performance margin derived from PS_fight.

That is how you convert round stats into better posterior updates.
